In [0]:
# Databricks notebook — Event Hubs -> bronze Delta table (Structured Streaming)
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

STORAGE = "stecomlakedev"
BRONZE  = f"abfss://bronze@{STORAGE}.dfs.core.windows.net/clickstream"
CKPT    = f"abfss://checkpoints@{STORAGE}.dfs.core.windows.net/bronze_clickstream"

conn = dbutils.secrets.get("ecomlake", "eventhub-listen-connstr")

# Kafka-compatible endpoint of Event Hubs (no extra library needed)
EH_NAMESPACE = "ehns-ecomlake-dev"
kafka_options = {
    "kafka.bootstrap.servers": f"{EH_NAMESPACE}.servicebus.windows.net:9093",
    "subscribe": "clickstream",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config":
        'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="$ConnectionString" password="{conn}";',
    "startingOffsets": "earliest",
}

raw = (spark.readStream.format("kafka").options(**kafka_options).load()
       .select(
           F.col("value").cast(StringType()).alias("raw_json"),
           F.col("timestamp").alias("ingest_ts"),
           F.col("partition"), F.col("offset"))
       .withColumn("ingest_date", F.to_date("ingest_ts")))

# Bronze principle: store the payload AS-IS. No parsing, no filtering.
q = (raw.writeStream
    .format("delta")
    .option("checkpointLocation", CKPT)
    .partitionBy("ingest_date")
    .trigger(availableNow=True)
    .start(BRONZE))

q.awaitTermination()
print("batch complete")